# Feature engineering

In [14]:
# Librerias 

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [35]:
# Caarga de datos 
try:
    path = "../data/raw/clinical_trial_data.csv"
    df = pd.read_csv(path)
    print(df.sample(10))
except:
    print("Error, datos no encontrados en la ruta ")




    Patient_ID  Age Gender   BMI  Systolic_BP  Glucose_Level Treatment_Arm  \
746    PT-0747   57      M  33.0        127.0          108.8        Drug_X   
813    PT-0814   57      F  20.3        117.0          107.3        Drug_X   
420    PT-0421   79      M  31.0        141.0          100.8        Drug_X   
766    PT-0767   54      F  23.9        115.0           95.2        Drug_X   
784    PT-0785   61      M  30.7        131.0          105.8       Placebo   
905    PT-0906   55      M  30.9        123.0          101.0        Drug_X   
732    PT-0733   59      M  27.4        134.0           93.3       Placebo   
163    PT-0164   45      F  28.7         96.0           95.1        Drug_X   
162    PT-0163   68      M  26.2        130.0          103.4        Drug_X   
88     PT-0089   48      F  34.0         97.0          117.4        Drug_X   

     Dropped_Out  
746            0  
813            0  
420            1  
766            1  
784            1  
905            0  
732     

In [19]:
# Ingeniería de datos 

X = df.drop(['Dropped_Out','Patient_ID'], axis = 1)
y = df['Dropped_Out']

In [20]:
# Separación y entrenamiento 

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state =42, stratify = y) 
print(f"Total de paciente, set de entrenamiento {len(X_train)}, set de prueba {len(X_test)}")

Total de paciente, set de entrenamiento 800, set de prueba 200


In [21]:
# Columnas numéricas y categóricas 

numeric_c = ['Age', 'BMI', 'Systolic_BP', 'Glucose_Level']
categorical_c = ['Gender', 'Treatment_Arm']

In [ ]:
# Preprocesamiento

preprocessor = ColumnTransformer(
    transformers= [
        ('num', StandardScaler(), numeric_c),
        ('cat', OneHotEncoder(drop = 'first', sparse_output = False), categorical_c)
    ], 
    remainder = 'passthrough'
)

In [28]:
# Tratamiento para evitar la fuga de datos 

X_train_pp = preprocessor.fit_transform(X_train)

X_test_pp = preprocessor.fit(X_test)

In [31]:
# Extracción de columnas para explicación clínica 

cat_features_n = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_c)
all_features_n = numeric_c + list(cat_features_n)

In [33]:
X_train_f = pd.DataFrame(X_train_pp, columns = all_features_n, index= X_train.index)
X_test_f = pd.DataFrame(X_test_pp, columns = all_features_n, index = X_test.index)

In [34]:
# MLOPs readiness 

joblib.dump(preprocessor, 'clinical_preprocessor.pkl')
print("Datos procesados correctamente\n")
print(X_train_f.head())

Datos procesados correctamente

          Age       BMI  Systolic_BP  Glucose_Level  Gender_M  \
492  0.012978  0.465292     1.995021      -0.770829       1.0   
30  -0.673475 -0.625255    -0.136390      -1.355300       1.0   
591 -0.501861 -0.347661    -0.710231      -0.280411       1.0   
585  0.184591  0.485120     0.027565      -3.175892       0.0   
897  0.098785  0.762713     1.175248      -0.683495       0.0   

     Treatment_Arm_Placebo  
492                    0.0  
30                     1.0  
591                    1.0  
585                    0.0  
897                    1.0  
